# MLOps Model Training Notebook: 30-Day Hospital Readmission Risk Prediction

This notebook implements the **train step** of the end-to-end MLOps pipeline:

```
generate_data.py → prep.py → train.py (THIS NOTEBOOK) → evaluate.py → register.py
```

## Purpose
- Load preprocessed train/validation datasets from `prep.py` outputs
- Train a GradientBoosting classifier using hyperparameters
- Log all parameters, metrics, and model artifacts to **MLflow** for experiment tracking
- Save the trained model with schema signature for serving validation

## Key MLOps Concepts
- **MLflow Tracking**: Records hyperparameters, metrics, and model artifacts for reproducibility and comparison across experiments
- **Data Pipeline**: Follows Azure ML component pattern (prep outputs → train inputs → evaluate inputs)
- **Promotion Gate**: Validation metrics are later evaluated in evaluate.py against thresholds before model registration

## Step 1: Environment Setup & Dependencies
Install required packages in the current notebook kernel.

### Why These Packages?
- **pandas + pyarrow**: Load/save Parquet files from prep.py outputs
- **scikit-learn**: GradientBoostingClassifier algorithm and evaluation metrics (accuracy, F1, precision, recall, AUC)
- **mlflow**: MLflow Tracking SDK - logs hyperparameters, metrics, and model artifacts to enable experiment comparison

### MLflow Role in Training:
MLflow tracks everything within a `run()` context, allowing you to:  
- Compare hyperparameter choices across experiments
- View metric evolution in MLflow UI
- Retrieve best model artifacts for deployment

In [ ]:
%pip install -q pandas numpy scikit-learn mlflow pyarrow

## Step 2: Imports & Shared Configuration

### What This Cell Does:
- Import ML libraries: sklearn (GradientBoostingClassifier, metrics), MLflow SDK
- Dynamically add `data_science/` to Python path
- Import `TARGET_COL = 'readmitted_30d'` from shared config

### Why Shared Config Matters:
The `config.py` module defines the **exact same target column** used by prep.py, train.py, and evaluate.py.  
This prevents column name mismatches that could break the pipeline or introduce bugs.  

### Shared Definitions in config.py:
- TARGET_COL: Binary label (0 = no readmission, 1 = readmitted within 30 days)
- NUMERIC_COLS: 12 continuous features (age, BMI, heart_rate_avg, etc.)
- CATEGORICAL_COLS: 5 categorical features (gender, admission_type, payer_code, etc.)

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_SCIENCE_DIR = PROJECT_ROOT / "data_science"
if str(DATA_SCIENCE_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_SCIENCE_DIR))

from config import TARGET_COL

print(f"Project root: {PROJECT_ROOT}")
print(f"Target column: {TARGET_COL}")

## Step 3: Configure Paths & Hyperparameters

### Local vs. Azure ML Paths:
In **production Azure ML**: The `train` component receives folder paths as input/output bindings (uri_folder type):
- prep.py → outputs train_data, val_data, test_data folders
- train.py → reads from those input folders  

**This notebook mimics that pattern locally:**
- Paths like `data/prepared/train_data/` and `data/prepared/val_data/` mirror Azure ML folder structure

### Hyperparameter Tuning (GradientBoostingClassifier):
These values are logged to MLflow and can be tuned to improve validation AUC:
- `n_estimators=300`: More boosting stages = better fit but higher risk of overfitting
- `max_depth=5`: Shallow trees for interpretability in healthcare models
- `learning_rate=0.1`: Shrinkage factor (smaller = more conservative, slower convergence)
- `min_samples_split=10`: Minimum samples to split a node (prevents overfitting)
- `subsample=0.8`: 80% of samples per iteration (adds regularization)

In [ ]:
raw_data_path = PROJECT_ROOT / "data" / "patients.csv"
prep_output_root = PROJECT_ROOT / "data" / "prepared"
train_data_dir = prep_output_root / "train_data"
val_data_dir = prep_output_root / "val_data"
test_data_dir = prep_output_root / "test_data"
model_output_dir = PROJECT_ROOT / "outputs" / "model"
model_output_dir.mkdir(parents=True, exist_ok=True)

params = {
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.1,
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "subsample": 0.8,
    "random_state": 42,
}

print("Raw data path:", raw_data_path)
print("Train data dir:", train_data_dir)
print("Val data dir:", val_data_dir)
print("Model output dir:", model_output_dir)
params

## Step 4: Data Preparation (Running Prep Component Locally)

### MLOps Pipeline Flow:
In **Azure ML in production**, this runs as a separate `prep` job before `train`. This cell automates that locally:

### Stage 1: generate_data.py
- Creates **synthetic patient data** (20,000 records, seed=42)
- Realistic distributions: age (normal), admission type (categorical), BMI, vitals, etc.
- Generates binary target using logistic model with domain-specific coefficients
- Output: `data/patients.csv`

### Stage 2: prep.py  
- **Load**: Read raw CSV
- **Clean**: Keep only valid features + target, drop NaN targets
- **Encode**: One-hot encode categorical features (5 categorical → 15+ encoded columns after drop_first=True)
- **Split**: **Stratified** 70/15/15 split (maintains readmission rate ~30% across all sets)
  - Train: fit the GradientBoostingClassifier
  - Val: tune hyperparameters and select best model
  - Test: held-out evaluation (used by evaluate.py, not visible to train.py)
- **Save**: Write parquet to folder outputs (Azure ML pattern)

### Why Folder Outputs?
Azure ML components use `uri_folder` for I/O (not individual files) to handle:
- Batch data (multiple parquets)
- Easy artifact versioning
- Seamless on local, AML, or cloud storage

### Output Artifacts:
- `data/prepared/train_data/train.parquet` → X_train, y_train
- `data/prepared/val_data/val.parquet` → X_val, y_val
- `data/prepared/test_data/test.parquet` → (held out for evaluate.py)

In [ ]:
import subprocess

generate_script = DATA_SCIENCE_DIR / "src" / "generate_data.py"
prep_script = DATA_SCIENCE_DIR / "src" / "prep.py"

if not raw_data_path.exists():
    print(f"Raw data not found at {raw_data_path}. Generating synthetic data...")
    raw_data_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            sys.executable,
            str(generate_script),
            "--output",
            str(raw_data_path),
            "--num_samples",
            "20000",
            "--seed",
            "42",
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        str(prep_script),
        "--raw_data",
        str(raw_data_path),
        "--train_data",
        str(train_data_dir),
        "--val_data",
        str(val_data_dir),
        "--test_data",
        str(test_data_dir),
    ],
    check=True,
 )

train_path = train_data_dir / "train.parquet"
val_path = val_data_dir / "val.parquet"

train = pd.read_parquet(train_path)
val = pd.read_parquet(val_path)

print("Train shape:", train.shape)
print("Val shape:", val.shape)
display(train.head(3))

## Step 5: Separate Features from Target Label

### Data Structure After Prep:
Each parquet file has shape (n_rows, 18):  
- **17 feature columns**: Numeric (age, BMI, heart_rate_avg, ...) + one-hot encoded categoricals
- **1 target column**: readmitted_30d (0 or 1)

### This Cell Splits Into:
- **X_train, X_val**: Feature matrices [n_samples × n_features]
- **y_train, y_val**: Target vectors [n_samples × 1]

### No Data Leakage:
The train/val split happens in `prep.py` with **stratification** (preserves class distribution).  
We never mix train and validation data → model only sees true train data during fitting.

In [ ]:
X_train = train.drop(columns=[TARGET_COL])
y_train = train[TARGET_COL]
X_val = val.drop(columns=[TARGET_COL])
y_val = val[TARGET_COL]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)

## Step 6: Train Model & Log Metrics to MLflow

### MLflow Context Manager Pattern:
```python
with mlflow.start_run(run_name="gradient_boosting_train"):
    # All logs here are recorded under this run
```

### Phase 1: Log Hyperparameters
- `mlflow.log_params(params)`: Records all GradientBoosting settings for reproducibility
- Enables: "Which hyperparameters led to best validation AUC?" queries in MLflow UI

### Phase 2: Train GradientBoostingClassifier
- `model.fit(X_train, y_train)`: Sequential boosting iterations on 70% of data
- **Binary classification**: predicts 0 (no readmission) or 1 (readmitted within 30 days)
- Learns patterns from patient features to predict readmission risk

### Phase 3: Compute Validation Metrics
**Train metrics** (model fit quality):
- `train_accuracy`: Fraction of correct predictions on training set
- `train_f1`: Harmonic mean of precision/recall

**Validation metrics** (generalization to unseen data):
- `val_accuracy`: Correct predictions on validation set (70-80% expected if no overfitting)
- `val_f1`: Balanced precision/recall
- `val_precision`: Of predicted readmissions, % are actually readmitted
- `val_recall`: Of actual readmissions, % are correctly identified
- `val_auc`: Area Under ROC Curve (0.5=random, 1.0=perfect) → **Key metric for imbalanced data**

### Phase 4: Log Metrics to MLflow
- `mlflow.log_metrics()`: Records all computed metrics
- These metrics flow to → evaluate.py (which applies promotion thresholds)

### Phase 5: Save Model with Schema
- `infer_signature(X_val, y_val_pred)`: Auto-captures feature types + output type
- `mlflow.sklearn.save_model()`: Saves model artifact with:
  - Fitted model parameters
  - Feature schema (for input validation at serving time)
  - input_example (5 sample rows for testing)

### Why MLflow Tracking?
- **Reproducibility**: Exact params + metrics recorded
- **Comparison**: Run 10 experiments with different hyperparameters, compare in UI
- **Lineage**: Track which data version led to which model performance
- **Serving**: Model Registry uses MLflow artifacts for deployment

In [ ]:
with mlflow.start_run(run_name="gradient_boosting_train"):
    mlflow.log_params(params)

    model = GradientBoostingClassifier(**params)
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_val_proba = model.predict_proba(X_val)[:, 1]

    train_metrics = {
        "train_accuracy": accuracy_score(y_train, y_train_pred),
        "train_f1": f1_score(y_train, y_train_pred),
    }

    val_metrics = {
        "val_accuracy": accuracy_score(y_val, y_val_pred),
        "val_f1": f1_score(y_val, y_val_pred),
        "val_precision": precision_score(y_val, y_val_pred),
        "val_recall": recall_score(y_val, y_val_pred),
        "val_auc": roc_auc_score(y_val, y_val_proba),
    }

    mlflow.log_metrics({**train_metrics, **val_metrics})

    signature = infer_signature(X_val, y_val_pred)
    input_example = X_val.head(5)
    mlflow.sklearn.save_model(
        sk_model=model,
        path=str(model_output_dir),
        signature=signature,
        input_example=input_example,
    )

    run_id = mlflow.active_run().info.run_id

print("Training metrics:", train_metrics)
print("Validation metrics:", val_metrics)
print(f"MLflow run_id: {run_id}")
print(f"Model saved to: {model_output_dir}")

## Step 7: Validation Metrics Summary

### Validation Set Performance:
Display metrics computed on the **unseen validation set** (15% of data).

### Promotion Gate Thresholds (Applied in evaluate.py):
The following minimums must be met on the **test set** to pass promotion:
- `test_auc >= 0.60`: AUC-ROC must exceed random guessing
- `test_f1 >= 0.40`: Balanced precision/recall threshold
- `test_precision >= 0.35`: At least 35% of predicted readmissions are correct
- `test_recall >= 0.35`: Catch at least 35% of true readmissions

If any threshold fails → `deploy_flag=0` → model does NOT get registered (register.py skips).
If all thresholds pass → `deploy_flag=1` → register.py registers model to Azure ML Model Registry.

### What Happens Next in the Pipeline:
1. **evaluate.py**: Loads this trained model + test.parquet (held-out 15%)
2. Computes test metrics and applies promotion gate thresholds
3. Outputs: deploy_flag (0 or 1), classification report, threshold_results.json
4. **register.py**: If deploy_flag=1, registers model to Azure ML Model Registry for serving

In [ ]:
pd.DataFrame([val_metrics]).T.rename(columns={0: "value"})

## Summary: Train Step in the Full MLOps Pipeline

### What We Completed (This Notebook):

**Data Flow:**
1. generate_data.py → synthetic patients.csv (20K records)
2. prep.py → clean, encode, stratified split → train/val/test parquets
3. **train.py (THIS NOTEBOOK)**
   - Load train/val parquets from prep.py outputs
   - Train GradientBoostingClassifier
   - Log hyperparameters + metrics to MLflow
   - Save model with schema signature

### Remaining Pipeline Steps:

**4. evaluate.py** (runs after train.py)
   - Loads trained model + test.parquet (held-out data not seen during training)
   - Computes test metrics (true generalization performance)
   - Applies promotion gate: checks val_auc >= 0.60, val_f1 >= 0.40, etc.
   - Outputs: deploy_flag (0 or 1), classification_report.json, probability_distribution.png

**5. register.py** (runs after evaluate.py)
   - If deploy_flag=1: Registers model to Azure ML Model Registry
   - If deploy_flag=0: Skips registration (model failed thresholds)
   - Outputs: model_info.json with version metadata

### Experimentation Locally:

**To experiment with hyperparameters:**
1. Edit Step 3 (modify n_estimators, max_depth, learning_rate)
2. Re-run Steps 5-7 to retrain and see new val metrics
3. Check if validation AUC improved

**To view all training runs:**
```bash
mlflow ui  # Open http://localhost:5000 in browser
```
Then compare runs side-by-side: hyperparameters vs. validation AUC.

### Production Deployment:
Once satisfied locally:
1. Push code to GitHub
2. GitHub Actions triggers Azure ML pipeline (multi-env-train.yml)
3. Pipeline runs: generate → prep → train → evaluate → register (all 5 steps)
4. Model automatically tested and registered if it passes thresholds
5. Deployment pipeline triggered (multi-env-deploy.yml) → test endpoint → integration tests → production endpoint